### Seção 1: Importações de Bibliotecas


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, when, split
from pyspark.sql.types import IntegerType, DecimalType
import os
import time
import psycopg2

### Seção 2: Inicialização da Sessão Spark


In [2]:
print("Iniciando a sessão Spark...")
spark = SparkSession.builder \
    .appName("Formula1_ETL_Raw_to_Database_v2") \
    .getOrCreate()
print("Sessão Spark iniciada com sucesso!")


Iniciando a sessão Spark...


Sessão Spark iniciada com sucesso!


### Seção 3: Definição de Caminhos


In [3]:
raw_path = "/home/jovyan/work/data"
print(f"Lendo arquivos da origem (RAW) de: {raw_path}")

print("Lendo arquivos da camada RAW com tratamento de nulos (\\N)...")
try:
    df_lap_times = spark.read.option("nullValue", "\\N").csv(os.path.join(raw_path, "lap_times.csv"), header=True, inferSchema=True)
    df_results = spark.read.option("nullValue", "\\N").csv(os.path.join(raw_path, "results.csv"), header=True, inferSchema=True)
    df_races = spark.read.option("nullValue", "\\N").csv(os.path.join(raw_path, "races.csv"), header=True, inferSchema=True)
    df_drivers = spark.read.option("nullValue", "\\N").csv(os.path.join(raw_path, "drivers.csv"), header=True, inferSchema=True)
    df_constructors = spark.read.option("nullValue", "\\N").csv(os.path.join(raw_path, "constructors.csv"), header=True, inferSchema=True)
    df_status = spark.read.option("nullValue", "\\N").csv(os.path.join(raw_path, "status.csv"), header=True, inferSchema=True)
    df_pit_stops = spark.read.option("nullValue", "\\N").csv(os.path.join(raw_path, "pit_stops.csv"), header=True, inferSchema=True)
    
    df_driver_standings = spark.read.option("nullValue", "\\N").csv(os.path.join(raw_path, "driver_standings.csv"), header=True, inferSchema=True)

    print("Arquivos RAW carregados com sucesso!")
except Exception as e:
    print(f"Erro ao ler os arquivos CSV. Verifique o caminho e a permissão dos arquivos. Erro: {e}")
    spark.stop()
    raise e

Lendo arquivos da origem (RAW) de: /home/jovyan/work/data
Lendo arquivos da camada RAW com tratamento de nulos (\N)...


Arquivos RAW carregados com sucesso!


### Seção 4: Leitura dos Dados Brutos (RAW)


In [4]:
print("Iniciando a transformação e unificação dos dados a partir dos resultados...")
print("Filtrando corridas para manter apenas de 2011 em diante (ou raceId >= 841)...")
races_silver = df_races.filter(
    (col("year") >= 2011) | (col("raceId") >= 841)
).select(
    col("raceId").alias("id_corrida"), col("year").alias("ano"),
    col("round").alias("rodada"), col("name").alias("nome_corrida")
)

drivers_silver = df_drivers.select(
    col("driverId").alias("id_piloto"), col("forename").alias("primeiro_nome_piloto"),
    col("surname").alias("sobrenome_piloto")
)
constructors_silver = df_constructors.select(
    col("constructorId").alias("id_equipe"), col("name").alias("nome_equipe")
)
status_silver = df_status.select(
    col("statusId").alias("id_status"), col("status").alias("descricao_status")
)

results_fact_base = df_results.select(
    col("raceId").alias("id_corrida"), col("driverId").alias("id_piloto"),
    col("constructorId").alias("id_equipe"), col("statusId").alias("id_status")
)

driver_standings_silver = df_driver_standings.select(
    col("raceId").alias("id_corrida"),
    col("driverId").alias("id_piloto"),
    col("points").alias("pontos_piloto"),
    col("position").alias("posicao_piloto"),
    col("wins").alias("vitorias_piloto")
)

min_tempo_ms = 53376 
max_tempo_ms = 300000 

Iniciando a transformação e unificação dos dados a partir dos resultados...
Filtrando corridas para manter apenas de 2011 em diante (ou raceId >= 841)...


### Seção 5: Transformação e Unificação dos Dados


In [5]:
df_lap_times_filtered = df_lap_times.filter(
    (col("milliseconds").isNotNull()) & 
    (col("milliseconds") >= min_tempo_ms) & 
    (col("milliseconds") <= max_tempo_ms)  
)

lap_times_detail = df_lap_times_filtered.select( 
    col("raceId").alias("id_corrida"), col("driverId").alias("id_piloto"),
    col("lap").cast(IntegerType()).alias("volta"),
    col("position").cast(IntegerType()).alias("posicao_na_volta"),
    col("milliseconds").cast(IntegerType()).alias("tempo_volta_ms")
)
print(f"Dados de voltas ('lap_times') filtrados por regras de negócio (min: {min_tempo_ms}ms, max: {max_tempo_ms}ms).")

pit_stops_cleaned = df_pit_stops.withColumn(
    "duracao_parada_seg",
    when(
        col("duration").contains(":"),
        split(col("duration"), ":").getItem(0).cast("double") * 60 + split(col("duration"), ":").getItem(1).cast("double")
    ).otherwise(
        expr("regexp_replace(duration, 's', '')").cast("double")
    )
).filter(col("duracao_parada_seg") <= 30).select(
    col("raceId").alias("id_corrida"), col("driverId").alias("id_piloto"),
    col("lap").alias("volta"), "duracao_parada_seg"
).na.drop(subset=["duracao_parada_seg"])

f1_unified_fact = results_fact_base.alias("res") \
    .join(races_silver.alias("race"), "id_corrida", "inner") \
    .join(drivers_silver.alias("driver"), "id_piloto", "left") \
    .join(constructors_silver.alias("const"), col("res.id_equipe") == col("const.id_equipe"), "left") \
    .join(status_silver.alias("stat"), col("res.id_status") == col("stat.id_status"), "left") \
    .join(lap_times_detail.alias("lap"), ["id_corrida", "id_piloto"], "left") \
    .join(pit_stops_cleaned.alias("pit"), ["id_corrida", "id_piloto", "volta"], "left")\
    .join(driver_standings_silver.alias("stand"), ["id_corrida", "id_piloto"], "left") 

f1_unified_fact = f1_unified_fact.select(
    col("const.id_equipe"),
    col("const.nome_equipe"),
    col("driver.id_piloto"),
    col("driver.primeiro_nome_piloto"),
    col("driver.sobrenome_piloto"),
    col("race.id_corrida"),
    col("lap.volta"),
    col("lap.posicao_na_volta"),
    col("lap.tempo_volta_ms"),
    col("stat.id_status"),
    col("stat.descricao_status"),
    col("pit.duracao_parada_seg"),
    
    col("stand.pontos_piloto"),
    #col("stand.posicao_piloto"),
    col("stand.vitorias_piloto"),
    
    col("race.ano"),
    col("race.rodada"),
    col("race.nome_corrida")
)

f1_unified_fact = f1_unified_fact.na.fill(0, subset=["duracao_parada_seg"])

f1_unified_fact = f1_unified_fact.na.fill(0, subset=["pontos_piloto", "vitorias_piloto"])

f1_unified_fact = f1_unified_fact.withColumn("duracao_parada_seg", col("duracao_parada_seg").cast(DecimalType(10, 3)))

print("Transformação de dados concluída!")
f1_unified_fact.printSchema()

Dados de voltas ('lap_times') filtrados por regras de negócio (min: 53376ms, max: 300000ms).


Transformação de dados concluída!
root
 |-- id_equipe: integer (nullable = true)
 |-- nome_equipe: string (nullable = true)
 |-- id_piloto: integer (nullable = true)
 |-- primeiro_nome_piloto: string (nullable = true)
 |-- sobrenome_piloto: string (nullable = true)
 |-- id_corrida: integer (nullable = true)
 |-- volta: integer (nullable = true)
 |-- posicao_na_volta: integer (nullable = true)
 |-- tempo_volta_ms: integer (nullable = true)
 |-- id_status: integer (nullable = true)
 |-- descricao_status: string (nullable = true)
 |-- duracao_parada_seg: decimal(10,3) (nullable = true)
 |-- pontos_piloto: double (nullable = false)
 |-- vitorias_piloto: integer (nullable = true)
 |-- ano: integer (nullable = true)
 |-- rodada: integer (nullable = true)
 |-- nome_corrida: string (nullable = true)



### Seção 6: Carga dos Dados no Banco de Dados PostgreSQL


In [6]:
print("\nIniciando a carga de dados no banco de dados...")
jdbc_hostname = os.getenv("DB_HOST")
jdbc_port     = os.getenv("DB_PORT")
jdbc_database = os.getenv("DB_NAME")
db_user       = os.getenv("DB_USER")
db_password   = os.getenv("DB_PASSWORD")
table_name    = "public.ResultadosCorridas"
jdbc_url = f"jdbc:postgresql://{jdbc_hostname}:{jdbc_port}/{jdbc_database}"
connection_properties = {"user": db_user, "password": db_password, "driver": "org.postgresql.Driver"}

retries = 10
wait_seconds = 5
for i in range(retries):
    try:
        print("Tentando conectar ao banco de dados...")
        conn = psycopg2.connect(host=jdbc_hostname, dbname=jdbc_database, user=db_user, password=db_password, port=jdbc_port)
        conn.close()
        print("✅ Conexão com o banco de dados bem-sucedida!")
        break
    except psycopg2.OperationalError as e:
        print(f"⏳ Banco de dados não está pronto. Tentando novamente em {wait_seconds} segundos...")
        time.sleep(wait_seconds)
        if i == retries - 1:
            print("❌ Não foi possível conectar ao banco de dados. Abortando.")
            spark.stop()
            raise e

try:
    print(f"Iniciando a escrita de dados na tabela: {table_name}")
    
    f1_unified_fact.write \
        .mode("append") \
        .jdbc(url=jdbc_url, table=table_name, properties=connection_properties)
        
    print(f"✅ Tabela '{table_name}' populada com sucesso no banco de dados '{jdbc_database}'!")

except Exception as e:
    print(f"❌ Ocorreu um erro ao salvar no banco de dados: {e}")
    spark.stop()
    raise e


Iniciando a carga de dados no banco de dados...
Tentando conectar ao banco de dados...
✅ Conexão com o banco de dados bem-sucedida!
Iniciando a escrita de dados na tabela: public.ResultadosCorridas


✅ Tabela 'public.ResultadosCorridas' populada com sucesso no banco de dados 'f1database'!


### Seção 7: Validação e Finalização


In [7]:
print("\nValidando a tabela fato principal 'f1_unified_fact' (amostra):")
f1_unified_fact.orderBy(col("ano").desc()).show(5, truncate=False) 

print("\n🚀 Job ETL (Raw → Database) finalizado com sucesso!")
spark.stop()


Validando a tabela fato principal 'f1_unified_fact' (amostra):


+---------+------------+---------+--------------------+----------------+----------+-----+----------------+--------------+---------+----------------+------------------+-------------+---------------+----+------+------------------+
|id_equipe|nome_equipe |id_piloto|primeiro_nome_piloto|sobrenome_piloto|id_corrida|volta|posicao_na_volta|tempo_volta_ms|id_status|descricao_status|duracao_parada_seg|pontos_piloto|vitorias_piloto|ano |rodada|nome_corrida      |
+---------+------------+---------+--------------------+----------------+----------+-----+----------------+--------------+---------+----------------+------------------+-------------+---------------+----+------+------------------+
|15       |Sauber      |855      |Guanyu              |Zhou            |1121      |1    |13              |104801        |11       |+1 Lap          |0.000             |0.0          |0              |2024|1     |Bahrain Grand Prix|
|210      |Haas F1 Team|807      |Nico                |Hülkenberg      |1121      |1